# Dataset Exploration & Quality Analysis

This notebook explores the generated penetration testing fine-tuning dataset.

In [ ]:
import json
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datasets.synthetic.generate_dataset import generate_dataset

# Generate dataset
dataset = generate_dataset(format='alpaca')
df = pd.DataFrame([
    {
        'instruction': d['instruction'],
        'output': d['output'],
        'category': d['metadata']['category'],
        'subcategory': d['metadata']['subcategory'],
        'instruction_len': len(d['instruction']),
        'output_len': len(d['output']),
        'has_code': '```' in d['output'],
    }
    for d in dataset
])
print(f'Total samples: {len(df)}')
df.head()

In [ ]:
# Category distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df['category'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Samples by Category')
axes[0].set_xlabel('')

df['output_len'].hist(ax=axes[1], bins=20, color='salmon', edgecolor='white')
axes[1].set_title('Response Length Distribution')
axes[1].set_xlabel('Characters')

df.groupby('category')['has_code'].mean().plot(kind='bar', ax=axes[2], color='green')
axes[2].set_title('Code Block Rate by Category')
axes[2].set_ylabel('Fraction')
axes[2].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('../output/dataset_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved to output/dataset_analysis.png')

In [ ]:
# Sample viewer
for _, row in df.sample(3, random_state=42).iterrows():
    print('='*60)
    print(f"Category: {row['category']} / {row['subcategory']}")
    print(f"Instruction: {row['instruction']}")
    print(f"Response length: {row['output_len']} chars | Code: {row['has_code']}")
    print('-'*60)
    print(row['output'][:500] + ('...' if len(row['output']) > 500 else ''))
    print()